# Build Customer Import From ACS Targets

This notebook creates a MySQL-ready customer import CSV for the synthetic Texas postal data warehouse.

`postal_customers.csv` provides the customer identity and assigned Texas location. The ACS ZCTA target file provides ZIP-level demographic probabilities for income, education, age, homeownership, and married-household status. The generated rows are synthetic and contain no real personal information.

This is a pre-PUMS approximation. Later PUMS donor sampling can replace some rule-based demographic generation while keeping the same customer import shape.

## 1. Imports and Configuration

Paths are built with `pathlib` and remain relative to the repository/project layout. The notebook is restartable and deterministic through `RANDOM_SEED`.

In [ ]:
from pathlib import Path
import hashlib
import math
import os
import random
import re
import uuid
from datetime import date, datetime, timedelta

import numpy as np
import pandas as pd

RANDOM_SEED = 42
DEMO_REFERENCE_DATE = date(2026, 1, 1)
PROJECT_ROOT = Path("Synthetic MySQL Tables") / "Customers"
if not PROJECT_ROOT.exists() and Path.cwd().name == "Customers":
    PROJECT_ROOT = Path(".")

OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAIN_OUTPUT_CSV = OUTPUT_DIR / "customer_import.csv"
AUDIT_OUTPUT_CSV = OUTPUT_DIR / "customer_import_audit.csv"
LOAD_SQL_PATH = OUTPUT_DIR / "load_customer_import.sql"

random.seed(RANDOM_SEED)
rng = np.random.default_rng(RANDOM_SEED)

print(f"Project root: {PROJECT_ROOT.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Random seed: {RANDOM_SEED}")

## 2. Load Inputs

The postal customer file supplies one row per synthetic/Olist-derived customer. The ACS target file supplies ZCTA-level control percentages.

In [ ]:
POSTAL_CUSTOMERS_CANDIDATES = [
    PROJECT_ROOT / "postal_customers.csv",
    PROJECT_ROOT / "raw" / "postal_customers.csv",
    Path("Synthetic_Customers_and_Orders") / "postal_customers.csv",
    Path("/mnt/data/postal_customers.csv"),
]

ACS_TARGET_CANDIDATES = [
    PROJECT_ROOT / "demographic profile pipeline" / "processed" / "census" / "tx_zcta_acs_targets_2024.csv",
    PROJECT_ROOT / "demographic profile pipeline" / "processed" / "census" / "tx_zcta_acs_targets_2024_audit.csv",
    Path("/mnt/data/tx_zcta_acs_targets_2024.csv"),
    Path("/mnt/data/tx_zcta_acs_targets_2024_audit.csv"),
]


def find_first_existing(candidates: list[Path], label: str) -> Path:
    for candidate in candidates:
        if candidate.exists():
            print(f"Using {label}: {candidate}")
            return candidate
    searched = "\n".join(f"  - {candidate}" for candidate in candidates)
    raise FileNotFoundError(f"Could not find {label}. Searched:\n{searched}")


postal_path = find_first_existing(POSTAL_CUSTOMERS_CANDIDATES, "postal_customers.csv")
acs_path = find_first_existing(ACS_TARGET_CANDIDATES, "ACS target file")

postal_raw = pd.read_csv(postal_path, dtype="string", low_memory=False)
acs_raw = pd.read_csv(acs_path, dtype="string", low_memory=False)

print(f"Postal rows: {len(postal_raw):,}; columns: {len(postal_raw.columns):,}")
print(f"ACS target rows: {len(acs_raw):,}; columns: {len(acs_raw.columns):,}")
display(postal_raw.head())
display(acs_raw.head())

## 3. Normalize Columns and Validate Inputs

Column names are inferred from likely aliases, then ZIP/ZCTA values are normalized as 5-character strings.

In [ ]:
POSTAL_COLUMN_CANDIDATES = {
    "source_customer_unique_id": ["customer_unique_id", "customer_id", "olist_customer_unique_id"],
    "source_order_count": ["order_count", "customer_order_count"],
    "source_first_customer_id": ["first_customer_id", "customer_id_first", "first_order_customer_id"],
    "zip_code": ["assigned_texas_zip", "zip_code", "assigned_zip", "zcta"],
    "city": ["assigned_texas_city", "city"],
    "assigned_texas_county": ["assigned_texas_county", "county"],
}

ACS_REQUIRED_COLUMNS = [
    "zcta",
    "total_households",
    "income_low_pct",
    "income_medium_pct",
    "income_high_pct",
    "income_very_high_pct",
    "owner_pct",
    "renter_pct",
    "married_household_pct",
    "partial_high_school_pct",
    "high_school_pct",
    "partial_college_pct",
    "bachelors_pct",
    "graduate_degree_pct",
    "age_18_24_pct",
    "age_25_34_pct",
    "age_35_44_pct",
    "age_45_64_pct",
    "age_65_plus_pct",
]


def resolve_column(frame: pd.DataFrame, candidates: list[str], logical_name: str, required: bool = True) -> str | None:
    normalized_lookup = {column.strip().lower(): column for column in frame.columns}
    for candidate in candidates:
        if candidate.lower() in normalized_lookup:
            return normalized_lookup[candidate.lower()]
    if required:
        raise ValueError(f"Missing required {logical_name} column. Tried: {candidates}")
    return None


def normalize_zip_series(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .str.strip()
        .str.extract(r"(\d{1,5})", expand=False)
        .fillna("")
        .str.zfill(5)
    )


postal_column_map = {
    logical_name: resolve_column(postal_raw, candidates, logical_name, required=(logical_name != "source_first_customer_id"))
    for logical_name, candidates in POSTAL_COLUMN_CANDIDATES.items()
}
print("Postal column map:")
for logical_name, source_column in postal_column_map.items():
    print(f"  {logical_name}: {source_column}")

missing_acs_columns = [column for column in ACS_REQUIRED_COLUMNS if column not in acs_raw.columns]
if missing_acs_columns:
    raise ValueError(f"ACS target file is missing required columns: {missing_acs_columns}")

postal = pd.DataFrame()
postal["source_customer_unique_id"] = postal_raw[postal_column_map["source_customer_unique_id"]].astype("string").str.strip()
postal["source_order_count"] = pd.to_numeric(postal_raw[postal_column_map["source_order_count"]], errors="coerce").fillna(0).astype(int)
postal["source_first_customer_id"] = (
    postal_raw[postal_column_map["source_first_customer_id"]].astype("string").str.strip()
    if postal_column_map["source_first_customer_id"]
    else pd.Series(pd.NA, index=postal_raw.index, dtype="string")
)
postal["zip_code"] = normalize_zip_series(postal_raw[postal_column_map["zip_code"]])
postal["zcta"] = postal["zip_code"]
postal["city"] = postal_raw[postal_column_map["city"]].astype("string").str.strip().str.title()
postal["assigned_texas_county"] = postal_raw[postal_column_map["assigned_texas_county"]].astype("string").str.strip()

duplicate_id_count = postal["source_customer_unique_id"].duplicated().sum()
if duplicate_id_count:
    print(f"Warning: found {duplicate_id_count:,} duplicate source customer IDs. Aggregating order_count and keeping first location row.")
    postal = (
        postal.sort_index()
        .groupby("source_customer_unique_id", dropna=False, as_index=False)
        .agg(
            source_order_count=("source_order_count", "sum"),
            source_first_customer_id=("source_first_customer_id", "first"),
            zip_code=("zip_code", "first"),
            zcta=("zcta", "first"),
            city=("city", "first"),
            assigned_texas_county=("assigned_texas_county", "first"),
        )
    )

acs = acs_raw.copy()
acs["zcta"] = normalize_zip_series(acs["zcta"])
for column in ACS_REQUIRED_COLUMNS:
    if column != "zcta":
        acs[column] = pd.to_numeric(acs[column], errors="coerce")

if acs["zcta"].duplicated().any():
    duplicate_zctas = acs.loc[acs["zcta"].duplicated(), "zcta"].dropna().unique()[:20].tolist()
    raise ValueError(f"ACS target file has duplicate zcta values. Examples: {duplicate_zctas}")

print(f"Normalized postal customers: {len(postal):,}")
print(f"Unique postal ZIPs: {postal['zcta'].nunique():,}")
print(f"Unique ACS ZCTAs: {acs['zcta'].nunique():,}")
display(postal.head())

## 4. Join ACS Targets to Customers

Customers without a local ACS row receive statewide-average target probabilities and are flagged in the audit output.

In [ ]:
PCT_COLUMNS = [column for column in ACS_REQUIRED_COLUMNS if column.endswith("_pct")]


def normalize_probability_columns(frame: pd.DataFrame, groups: list[list[str]]) -> pd.DataFrame:
    normalized = frame.copy()
    for columns in groups:
        values = normalized[columns].apply(pd.to_numeric, errors="coerce")
        row_sums = values.sum(axis=1, skipna=True)
        valid = row_sums > 0
        normalized.loc[valid, columns] = values.loc[valid, columns].div(row_sums[valid], axis=0)
    return normalized


acs_statewide_average = acs[PCT_COLUMNS + ["total_households"]].copy()
weights = pd.to_numeric(acs_statewide_average["total_households"], errors="coerce").fillna(0)
statewide_values = {}
for column in PCT_COLUMNS:
    series = pd.to_numeric(acs_statewide_average[column], errors="coerce")
    if weights.sum() > 0 and series.notna().any():
        statewide_values[column] = np.average(series.fillna(series.mean()), weights=weights)
    else:
        statewide_values[column] = series.mean()
statewide_values["total_households"] = weights.mean()

customers = postal.merge(acs[ACS_REQUIRED_COLUMNS], on="zcta", how="left", indicator=True)
customers["acs_match_status"] = np.where(customers["_merge"].eq("both"), "matched_zcta", "fallback_statewide_average")
customers = customers.drop(columns=["_merge"])

missing_mask = customers["acs_match_status"].eq("fallback_statewide_average")
for column, value in statewide_values.items():
    customers.loc[missing_mask, column] = customers.loc[missing_mask, column].fillna(value)

customers = normalize_probability_columns(
    customers,
    [
        ["income_low_pct", "income_medium_pct", "income_high_pct", "income_very_high_pct"],
        ["owner_pct", "renter_pct"],
        ["partial_high_school_pct", "high_school_pct", "partial_college_pct", "bachelors_pct", "graduate_degree_pct"],
        ["age_18_24_pct", "age_25_34_pct", "age_35_44_pct", "age_45_64_pct", "age_65_plus_pct"],
    ],
)
customers["married_household_pct"] = pd.to_numeric(customers["married_household_pct"], errors="coerce").clip(0, 1).fillna(0.5)

missing_zips = sorted(customers.loc[missing_mask, "zcta"].dropna().unique().tolist())
print(f"Total customers: {len(customers):,}")
print(f"Matched to ACS target row: {(~missing_mask).sum():,}")
print(f"Missing ACS target row: {missing_mask.sum():,}")
print(f"Missing ACS ZIP count: {len(missing_zips):,}")
if missing_zips:
    print(f"Missing ACS ZIP examples: {missing_zips[:25]}")

## 5. Helper Functions

These helpers keep ID normalization, weighted assignment, and synthetic field generation isolated and testable.

In [ ]:
HEX_32_RE = re.compile(r"^[0-9a-f]{32}$")


def normalize_hex_customer_id(value, fallback_key: str) -> tuple[str, bool]:
    raw_value = "" if pd.isna(value) else str(value).strip().lower().replace("-", "")
    if HEX_32_RE.fullmatch(raw_value):
        return raw_value, False
    fallback_uuid = uuid.uuid5(uuid.NAMESPACE_URL, f"synthetic-texas-postal-customer:{fallback_key}")
    return fallback_uuid.hex, True


def weighted_choice(labels: list[str], weights, rng: np.random.Generator) -> str:
    probabilities = np.asarray([0 if pd.isna(weight) else float(weight) for weight in weights], dtype=float)
    probabilities = np.where(probabilities < 0, 0, probabilities)
    if probabilities.sum() <= 0:
        probabilities = np.repeat(1 / len(labels), len(labels))
    else:
        probabilities = probabilities / probabilities.sum()
    return str(rng.choice(labels, p=probabilities))


def make_exact_category_assignments(n: int, labels: list[str], probabilities, rng: np.random.Generator) -> np.ndarray:
    if n <= 0:
        return np.array([], dtype=object)
    probs = np.asarray([0 if pd.isna(probability) else float(probability) for probability in probabilities], dtype=float)
    probs = np.where(probs < 0, 0, probs)
    if probs.sum() <= 0:
        probs = np.repeat(1 / len(labels), len(labels))
    else:
        probs = probs / probs.sum()
    raw_counts = probs * n
    counts = np.floor(raw_counts).astype(int)
    remainder = n - counts.sum()
    if remainder > 0:
        order = np.argsort(-(raw_counts - counts))
        counts[order[:remainder]] += 1
    assignments = np.concatenate([np.repeat(label, count) for label, count in zip(labels, counts)])
    rng.shuffle(assignments)
    return assignments


def generate_income_from_band(band: str, rng: np.random.Generator) -> float:
    ranges = {
        "Low": (18000, 49999, 33000),
        "Medium": (50000, 99999, 72000),
        "High": (100000, 149999, 118000),
        "Very High": (150000, 250000, 180000),
    }
    low, high, mode = ranges.get(band, ranges["Medium"])
    value = rng.triangular(low, mode, high)
    return round(float(value), 2)


def generate_age_from_band(age_band: str, rng: np.random.Generator) -> int:
    ranges = {
        "18_24": (18, 24),
        "25_34": (25, 34),
        "35_44": (35, 44),
        "45_64": (45, 64),
        "65_plus": (65, 90),
    }
    low, high = ranges.get(age_band, ranges["25_34"])
    return int(rng.integers(low, high + 1))


def subtract_years(value: date, years: int) -> date:
    try:
        return value.replace(year=value.year - years)
    except ValueError:
        return value.replace(month=2, day=28, year=value.year - years)


def generate_birth_date_from_age(age: int, rng: np.random.Generator) -> str:
    earliest = subtract_years(DEMO_REFERENCE_DATE, age + 1) + timedelta(days=1)
    latest = subtract_years(DEMO_REFERENCE_DATE, age)
    offset_days = int(rng.integers(0, (latest - earliest).days + 1))
    return (earliest + timedelta(days=offset_days)).isoformat()


def generate_gender(rng: np.random.Generator) -> str:
    return weighted_choice(["F", "M"], [0.51, 0.49], rng)


def generate_occupation(education_level: str, income_band: str, age: int, rng: np.random.Generator) -> str:
    labels = ["Manual", "Clerical", "Management", "Professional", "Skilled Manual"]
    weights = np.array([0.20, 0.20, 0.18, 0.22, 0.20], dtype=float)
    if education_level == "Graduate Degree":
        weights = np.array([0.04, 0.10, 0.32, 0.45, 0.09])
    elif education_level == "Bachelors":
        weights = np.array([0.06, 0.18, 0.28, 0.38, 0.10])
    elif education_level == "Partial College":
        weights = np.array([0.15, 0.30, 0.14, 0.20, 0.21])
    elif education_level == "High School":
        weights = np.array([0.25, 0.24, 0.10, 0.10, 0.31])
    elif education_level == "Partial High School":
        weights = np.array([0.45, 0.12, 0.04, 0.04, 0.35])
    if income_band in {"High", "Very High"}:
        weights += np.array([-0.04, -0.02, 0.10, 0.10, -0.04])
    if age >= 55 and income_band in {"High", "Very High"}:
        weights += np.array([-0.02, -0.03, 0.08, 0.01, -0.04])
    weights = np.clip(weights, 0.01, None)
    return weighted_choice(labels, weights, rng)


def generate_total_children(age: int, marital_status: str, income_band: str, rng: np.random.Generator) -> int:
    if age < 25:
        labels, weights = [0, 1, 2, 3, 4, 5], [0.86, 0.10, 0.03, 0.01, 0.00, 0.00]
    elif age <= 44 and marital_status == "M":
        labels, weights = [0, 1, 2, 3, 4, 5], [0.28, 0.25, 0.25, 0.15, 0.05, 0.02]
    elif age <= 44:
        labels, weights = [0, 1, 2, 3, 4, 5], [0.66, 0.20, 0.10, 0.03, 0.01, 0.00]
    elif age <= 64 and marital_status == "M":
        labels, weights = [0, 1, 2, 3, 4, 5], [0.55, 0.20, 0.15, 0.07, 0.02, 0.01]
    elif age <= 64:
        labels, weights = [0, 1, 2, 3, 4, 5], [0.78, 0.14, 0.06, 0.02, 0.00, 0.00]
    else:
        labels, weights = [0, 1, 2, 3, 4, 5], [0.94, 0.04, 0.02, 0.00, 0.00, 0.00]
    if income_band == "Very High" and 25 <= age <= 54 and marital_status == "M":
        weights = np.asarray(weights, dtype=float) + np.array([-0.03, 0.00, 0.02, 0.01, 0.00, 0.00])
    return int(weighted_choice([str(label) for label in labels], weights, rng))


FEMALE_FIRST_NAMES = [
    "Alyssa", "Camila", "Danielle", "Elena", "Gabriela", "Hannah", "Isabel", "Jasmine", "Kayla", "Lauren",
    "Maya", "Natalie", "Olivia", "Priya", "Rachel", "Sofia", "Tara", "Valerie", "Whitney", "Zoe",
]
MALE_FIRST_NAMES = [
    "Aaron", "Caleb", "Daniel", "Ethan", "Felix", "Gabriel", "Isaac", "Julian", "Kevin", "Luis",
    "Marcus", "Nathan", "Omar", "Preston", "Rafael", "Samuel", "Thomas", "Victor", "Wesley", "Xavier",
]
LAST_NAMES = [
    "Anderson", "Bennett", "Castillo", "Diaz", "Edwards", "Flores", "Garcia", "Hayes", "Johnson", "Kim",
    "Lee", "Martinez", "Nguyen", "Ortiz", "Patel", "Ramirez", "Singh", "Taylor", "Vasquez", "Walker",
]
STREET_NAMES = [
    "Bluebonnet", "Cedar Creek", "Live Oak", "Mesquite", "Pecan Grove", "Rio Vista", "Lone Star", "Hill Country",
    "Prairie Bend", "Sunset Ridge", "Trinity", "Brazos", "Mockingbird", "Cypress", "Magnolia", "Post Oak",
]
STREET_SUFFIXES = ["St", "Ave", "Dr", "Ln", "Rd", "Ct", "Way", "Blvd"]
TEXAS_AREA_CODES = ["214", "469", "972", "817", "682", "713", "281", "832", "346", "512", "737", "210", "726", "915", "903", "936", "979", "956", "940", "325", "432", "361", "806", "830"]


def generate_name(gender: str, rng: np.random.Generator) -> tuple[str, str, str]:
    first_names = FEMALE_FIRST_NAMES if gender == "F" else MALE_FIRST_NAMES
    first = str(rng.choice(first_names))
    last = str(rng.choice(LAST_NAMES))
    middle = "" if rng.random() < 0.18 else chr(int(rng.integers(ord("A"), ord("Z") + 1)))
    return first, middle, last


def generate_street_address(rng: np.random.Generator) -> str:
    number = int(rng.integers(101, 9899))
    return f"{number} {rng.choice(STREET_NAMES)} {rng.choice(STREET_SUFFIXES)}"


def generate_phone_number(zip_code: str, rng: np.random.Generator) -> str:
    metro_area_codes = {
        "75": ["214", "469", "972", "817", "682"],
        "76": ["817", "682", "940", "325"],
        "77": ["713", "281", "832", "346", "936", "979"],
        "78": ["512", "737", "210", "726", "830"],
        "79": ["915", "432", "806", "325"],
    }
    area_codes = metro_area_codes.get(str(zip_code)[:2], TEXAS_AREA_CODES)
    area_code = str(rng.choice(area_codes))
    line_number = int(rng.integers(0, 10000))
    return f"{area_code}-555-{line_number:04d}"


def safe_email_token(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", ".", value.lower()).strip(".")


def generate_email(first_name: str, last_name: str, customer_id: str, existing_emails: set[str]) -> str:
    base = f"{safe_email_token(first_name)}.{safe_email_token(last_name)}.{customer_id[:8]}"
    email = f"{base}@postal-demo.example"
    counter = 2
    while email in existing_emails:
        email = f"{base}.{counter}@postal-demo.example"
        counter += 1
    existing_emails.add(email)
    return email


def generated_timestamp(row_number: int) -> str:
    base = datetime(2024, 1, 1, 8, 0, 0)
    timestamp = base + timedelta(minutes=(row_number * 17) % (366 * 24 * 60))
    return timestamp.strftime("%Y-%m-%d %H:%M:%S")

## 6. ZIP-Group Demographic Assignment

Key ACS-controlled fields are assigned by ZIP group using exact category counts. This keeps each larger ZIP closer to its target distribution than independent row-by-row draws.

In [ ]:
INCOME_LABELS = ["Low", "Medium", "High", "Very High"]
INCOME_COLUMNS = ["income_low_pct", "income_medium_pct", "income_high_pct", "income_very_high_pct"]
EDUCATION_LABELS = ["Partial High School", "High School", "Partial College", "Bachelors", "Graduate Degree"]
EDUCATION_COLUMNS = ["partial_high_school_pct", "high_school_pct", "partial_college_pct", "bachelors_pct", "graduate_degree_pct"]
AGE_LABELS = ["18_24", "25_34", "35_44", "45_64", "65_plus"]
AGE_COLUMNS = ["age_18_24_pct", "age_25_34_pct", "age_35_44_pct", "age_45_64_pct", "age_65_plus_pct"]
HOME_OWNER_LABELS = ["Y", "N"]
HOME_OWNER_COLUMNS = ["owner_pct", "renter_pct"]
MARITAL_LABELS = ["M", "S"]

generated_parts = []


def target_probabilities(target_row: pd.Series, columns: list[str]) -> np.ndarray:
    return pd.to_numeric(target_row[columns], errors="coerce").to_numpy(dtype=float)

for zcta, group in customers.groupby("zcta", sort=True):
    group_generated = group.copy()
    n = len(group_generated)
    target = group_generated.iloc[0]

    group_generated["income_band"] = make_exact_category_assignments(n, INCOME_LABELS, target_probabilities(target, INCOME_COLUMNS), rng)
    group_generated["education_level"] = make_exact_category_assignments(n, EDUCATION_LABELS, target_probabilities(target, EDUCATION_COLUMNS), rng)
    group_generated["age_band"] = make_exact_category_assignments(n, AGE_LABELS, target_probabilities(target, AGE_COLUMNS), rng)
    group_generated["home_owner"] = make_exact_category_assignments(n, HOME_OWNER_LABELS, target_probabilities(target, HOME_OWNER_COLUMNS), rng)
    married_probability = float(target["married_household_pct"]) if not pd.isna(target["married_household_pct"]) else 0.5
    group_generated["marital_status"] = make_exact_category_assignments(n, MARITAL_LABELS, [married_probability, 1 - married_probability], rng)
    generated_parts.append(group_generated)

generated = pd.concat(generated_parts, ignore_index=True)

customer_ids = []
customer_id_fallback_flags = []
for row_number, row in generated.iterrows():
    fallback_key = "|".join([
        str(row_number),
        str(row.get("source_customer_unique_id", "")),
        str(row.get("source_first_customer_id", "")),
        str(row.get("zip_code", "")),
    ])
    customer_id, was_fallback = normalize_hex_customer_id(row["source_customer_unique_id"], fallback_key)
    customer_ids.append(customer_id)
    customer_id_fallback_flags.append(was_fallback)

generated["customer_id"] = customer_ids
generated["customer_id_was_generated_fallback"] = customer_id_fallback_flags

duplicate_customer_ids = generated["customer_id"].duplicated(keep=False)
if duplicate_customer_ids.any():
    print(f"Warning: {duplicate_customer_ids.sum():,} generated customer IDs were duplicated. Replacing duplicate occurrences with deterministic UUID fallbacks.")
    duplicate_occurrence = generated.groupby("customer_id").cumcount()
    duplicate_fix_mask = duplicate_occurrence > 0
    for idx in generated.index[duplicate_fix_mask]:
        fallback_key = f"duplicate-customer-id:{idx}:{generated.at[idx, 'customer_id']}"
        generated.at[idx, "customer_id"] = uuid.uuid5(uuid.NAMESPACE_URL, fallback_key).hex
        generated.at[idx, "customer_id_was_generated_fallback"] = True

existing_emails: set[str] = set()
records = []

for row_number, row in generated.reset_index(drop=True).iterrows():
    age = generate_age_from_band(row["age_band"], rng)
    gender = generate_gender(rng)
    first_name, middle_initial, last_name = generate_name(gender, rng)
    email = generate_email(first_name, last_name, row["customer_id"], existing_emails)
    created_at = generated_timestamp(row_number)
    annual_income = generate_income_from_band(row["income_band"], rng)
    occupation = generate_occupation(row["education_level"], row["income_band"], age, rng)
    total_children = generate_total_children(age, row["marital_status"], row["income_band"], rng)

    records.append(
        {
            "customer_id": row["customer_id"],
            "first_name": first_name,
            "middle_initial": middle_initial,
            "last_name": last_name,
            "street_address": generate_street_address(rng),
            "city": row["city"],
            "state_code": "TX",
            "zip_code": row["zip_code"],
            "territory_id": "",
            "phone_number": generate_phone_number(row["zip_code"], rng),
            "email": email,
            "created_at": created_at,
            "updated_at": created_at,
            "user_id": "",
            "preferred_facility_id": "",
            "birth_date": generate_birth_date_from_age(age, rng),
            "marital_status": row["marital_status"],
            "gender": gender,
            "email_address": email,
            "annual_income": f"{annual_income:.2f}",
            "total_children": int(total_children),
            "education_level": row["education_level"],
            "occupation": occupation,
            "home_owner": row["home_owner"],
            "source_customer_unique_id": row["source_customer_unique_id"],
            "source_first_customer_id": row["source_first_customer_id"],
            "source_order_count": row["source_order_count"],
            "assigned_texas_county": row["assigned_texas_county"],
            "zcta": row["zcta"],
            "acs_match_status": row["acs_match_status"],
            "customer_id_was_generated_fallback": bool(row["customer_id_was_generated_fallback"]),
            "income_band": row["income_band"],
            "age_band": row["age_band"],
            "generated_age": age,
            "generation_seed": RANDOM_SEED,
            "demographic_source": "acs_zcta_targets_rule_based",
        }
    )

generated_output = pd.DataFrame(records)

print(f"Generated customer rows: {len(generated_output):,}")
print(f"Generated unique emails: {generated_output['email'].nunique():,}")
print(f"Generated fallback customer IDs: {generated_output['customer_id_was_generated_fallback'].sum():,}")
display(generated_output.head())

## 7. Build Final Main and Audit Tables

The main CSV contains only table-aligned customer columns. Debugging and provenance fields stay in the audit CSV.

In [ ]:
CUSTOMER_IMPORT_COLUMNS = [
    "customer_id",
    "first_name",
    "middle_initial",
    "last_name",
    "street_address",
    "city",
    "state_code",
    "zip_code",
    "territory_id",
    "phone_number",
    "email",
    "created_at",
    "updated_at",
    "user_id",
    "preferred_facility_id",
    "birth_date",
    "marital_status",
    "gender",
    "email_address",
    "annual_income",
    "total_children",
    "education_level",
    "occupation",
    "home_owner",
]

AUDIT_EXTRA_COLUMNS = [
    "source_customer_unique_id",
    "source_first_customer_id",
    "source_order_count",
    "assigned_texas_county",
    "zcta",
    "acs_match_status",
    "customer_id_was_generated_fallback",
    "income_band",
    "age_band",
    "generated_age",
    "generation_seed",
    "demographic_source",
]

customer_import = generated_output[CUSTOMER_IMPORT_COLUMNS].copy()
customer_import["city"] = customer_import["city"].astype("string").str.title()
customer_import["zip_code"] = normalize_zip_series(customer_import["zip_code"])
customer_import["customer_id"] = customer_import["customer_id"].astype("string").str.lower()
customer_import["annual_income"] = pd.to_numeric(customer_import["annual_income"], errors="coerce").map(lambda value: f"{value:.2f}")
customer_import["total_children"] = pd.to_numeric(customer_import["total_children"], errors="coerce").astype(int)

customer_import_audit = pd.concat(
    [customer_import, generated_output[AUDIT_EXTRA_COLUMNS].reset_index(drop=True)],
    axis=1,
)

print(f"Main columns: {list(customer_import.columns)}")
print(f"Audit columns: {list(customer_import_audit.columns)}")
display(customer_import.head())

## 8. Validation

Critical compatibility issues fail immediately. Distribution-quality checks print summary metrics for ZIP groups with at least 50 customers.

In [ ]:
def require(condition: bool, message: str) -> None:
    if not bool(condition):
        raise AssertionError(message)


def calculate_age_on_reference(birth_date_value: str) -> int:
    parsed = datetime.strptime(str(birth_date_value), "%Y-%m-%d").date()
    return DEMO_REFERENCE_DATE.year - parsed.year - ((DEMO_REFERENCE_DATE.month, DEMO_REFERENCE_DATE.day) < (parsed.month, parsed.day))


REQUIRED_NON_NULL_COLUMNS = [
    "customer_id",
    "first_name",
    "last_name",
    "street_address",
    "city",
    "state_code",
    "zip_code",
    "phone_number",
    "email",
]
ALLOWED_EDUCATION = set(EDUCATION_LABELS)
ALLOWED_OCCUPATION = {"Manual", "Clerical", "Management", "Professional", "Skilled Manual"}

require(customer_import["customer_id"].is_unique, "customer_id must be unique")
require(customer_import["customer_id"].str.fullmatch(r"[0-9a-f]{32}").all(), "customer_id must be exactly 32 lowercase hex characters")
require(customer_import["email"].is_unique, "email must be unique")
require(customer_import["state_code"].eq("TX").all(), "state_code must be TX")
require(customer_import["zip_code"].str.fullmatch(r"\d{5}").all(), "zip_code must be exactly 5 digits")
require(customer_import["marital_status"].isin(["M", "S"]).all(), "marital_status must be M or S")
require(customer_import["gender"].isin(["M", "F"]).all(), "gender must be M or F")
require(customer_import["home_owner"].isin(["Y", "N"]).all(), "home_owner must be Y or N")
require(customer_import["total_children"].between(0, 5).all(), "total_children must be between 0 and 5")
require(customer_import["education_level"].isin(ALLOWED_EDUCATION).all(), "education_level contains values outside allowed list")
require(customer_import["occupation"].isin(ALLOWED_OCCUPATION).all(), "occupation contains values outside allowed list")
require((pd.to_numeric(customer_import["annual_income"], errors="coerce") >= 0).all(), "annual_income must be non-negative")
require(customer_import[REQUIRED_NON_NULL_COLUMNS].replace("", pd.NA).notna().all().all(), "Required non-null customer fields cannot be blank")

ages = customer_import["birth_date"].map(calculate_age_on_reference)
require((ages >= 18).all(), "All customers must be at least 18 as of DEMO_REFERENCE_DATE")

print("Critical customer import validations passed.")


def compare_distribution(generated_frame: pd.DataFrame, group_name: str, labels: list[str], generated_column: str, target_columns: list[str], min_zip_size: int = 50) -> pd.DataFrame:
    rows = []
    for zcta, zip_group in generated_frame.groupby("zcta"):
        if len(zip_group) < min_zip_size:
            continue
        target_row = customers.loc[customers["zcta"].eq(zcta)].iloc[0]
        actual = zip_group[generated_column].value_counts(normalize=True).reindex(labels, fill_value=0)
        if group_name == "marital_status":
            married_probability = float(pd.to_numeric(pd.Series([target_row["married_household_pct"]]), errors="coerce").fillna(0.5).iloc[0])
            target = pd.Series([married_probability, 1 - married_probability], index=labels)
        else:
            target_values = pd.to_numeric(target_row[target_columns], errors="coerce").fillna(0).to_numpy(dtype=float)
            if target_values.sum() <= 0:
                target_values = np.repeat(1 / len(labels), len(labels))
            else:
                target_values = target_values / target_values.sum()
            target = pd.Series(target_values, index=labels)
        abs_errors = (actual - target).abs() * 100
        rows.append(
            {
                "zcta": zcta,
                "group": group_name,
                "customer_count": len(zip_group),
                "mean_abs_pct_point_error": abs_errors.mean(),
                "max_abs_pct_point_error": abs_errors.max(),
            }
        )
    return pd.DataFrame(rows)


distribution_frames = [
    compare_distribution(customer_import_audit, "income", INCOME_LABELS, "income_band", INCOME_COLUMNS),
    compare_distribution(customer_import_audit, "education", EDUCATION_LABELS, "education_level", EDUCATION_COLUMNS),
    compare_distribution(customer_import_audit, "home_owner", HOME_OWNER_LABELS, "home_owner", HOME_OWNER_COLUMNS),
    compare_distribution(customer_import_audit, "marital_status", MARITAL_LABELS, "marital_status", ["married_household_pct", "married_household_pct"]),
    compare_distribution(customer_import_audit, "age", AGE_LABELS, "age_band", AGE_COLUMNS),
]
distribution_quality = pd.concat(distribution_frames, ignore_index=True)

if not distribution_quality.empty:
    print("Mean absolute percentage point error by category group:")
    display(distribution_quality.groupby("group", as_index=False)["mean_abs_pct_point_error"].mean().sort_values("group"))
    print("Worst 10 ZIP/category groups by demographic mismatch:")
    display(distribution_quality.sort_values("mean_abs_pct_point_error", ascending=False).head(10))
else:
    print("No ZIP groups with at least 50 customers for distribution-quality reporting.")

## 9. Save Outputs

The CSV files use UTF-8 encoding and omit DataFrame indexes.

In [ ]:
customer_import.to_csv(MAIN_OUTPUT_CSV, index=False, encoding="utf-8")
customer_import_audit.to_csv(AUDIT_OUTPUT_CSV, index=False, encoding="utf-8")

print(f"Saved main customer import: {MAIN_OUTPUT_CSV.resolve()} ({len(customer_import):,} rows)")
print(f"Saved audit customer import: {AUDIT_OUTPUT_CSV.resolve()} ({len(customer_import_audit):,} rows)")

## 10. Generate Helper SQL Load Script

The staging table keeps `customer_id` as hex text. The final insert converts it to `BINARY(16)` with `UNHEX(customer_id)`.

In [ ]:
LOAD_SQL = """
-- Helper load script for Synthetic MySQL Tables/Customers/output/customer_import.csv
-- customer.customer_id is BINARY(16), so the CSV stores lowercase hex text
-- and this script converts it with UNHEX(customer_id) during insert.

DROP TABLE IF EXISTS staging_customer_import;

CREATE TABLE staging_customer_import (
    customer_id CHAR(32) NOT NULL,
    first_name VARCHAR(50) NOT NULL,
    middle_initial CHAR(1) NULL,
    last_name VARCHAR(50) NOT NULL,
    street_address VARCHAR(100) NOT NULL,
    city VARCHAR(50) NOT NULL,
    state_code CHAR(2) NOT NULL,
    zip_code VARCHAR(10) NOT NULL,
    territory_id VARCHAR(20) NULL,
    phone_number VARCHAR(15) NOT NULL,
    email VARCHAR(100) NOT NULL,
    created_at VARCHAR(30) NULL,
    updated_at VARCHAR(30) NULL,
    user_id VARCHAR(20) NULL,
    preferred_facility_id VARCHAR(20) NULL,
    birth_date VARCHAR(20) NULL,
    marital_status CHAR(1) NULL,
    gender CHAR(1) NULL,
    email_address VARCHAR(150) NULL,
    annual_income VARCHAR(30) NULL,
    total_children VARCHAR(10) NULL,
    education_level VARCHAR(30) NULL,
    occupation VARCHAR(30) NULL,
    home_owner CHAR(1) NULL
);

-- Update this placeholder path before running.
LOAD DATA LOCAL INFILE '/absolute/path/to/customer_import.csv'
INTO TABLE staging_customer_import
CHARACTER SET utf8mb4
FIELDS TERMINATED BY ',' ENCLOSED BY '"' ESCAPED BY '"'
LINES TERMINATED BY '\n'
IGNORE 1 LINES
(
    customer_id,
    first_name,
    middle_initial,
    last_name,
    street_address,
    city,
    state_code,
    zip_code,
    territory_id,
    phone_number,
    email,
    created_at,
    updated_at,
    user_id,
    preferred_facility_id,
    birth_date,
    marital_status,
    gender,
    email_address,
    annual_income,
    total_children,
    education_level,
    occupation,
    home_owner
);

INSERT INTO customer (
    customer_id,
    first_name,
    middle_initial,
    last_name,
    street_address,
    city,
    state_code,
    zip_code,
    territory_id,
    phone_number,
    email,
    created_at,
    updated_at,
    user_id,
    preferred_facility_id,
    birth_date,
    marital_status,
    gender,
    email_address,
    annual_income,
    total_children,
    education_level,
    occupation,
    home_owner
)
SELECT
    UNHEX(customer_id),
    first_name,
    NULLIF(middle_initial, ''),
    last_name,
    street_address,
    city,
    state_code,
    zip_code,
    NULLIF(territory_id, ''),
    phone_number,
    email,
    NULLIF(created_at, ''),
    NULLIF(updated_at, ''),
    NULLIF(user_id, ''),
    NULLIF(preferred_facility_id, ''),
    NULLIF(birth_date, ''),
    NULLIF(marital_status, ''),
    NULLIF(gender, ''),
    NULLIF(email_address, ''),
    NULLIF(annual_income, ''),
    NULLIF(total_children, ''),
    NULLIF(education_level, ''),
    NULLIF(occupation, ''),
    NULLIF(home_owner, '')
FROM staging_customer_import;
"""

LOAD_SQL_PATH.write_text(LOAD_SQL.strip() + "\n", encoding="utf-8")
print(f"Saved helper SQL load script: {LOAD_SQL_PATH.resolve()}")

## 11. Display Examples and Summary Tables

In [ ]:
print("First 10 customer import rows:")
display(customer_import.head(10))

print("Audit sample:")
display(customer_import_audit.head(10))

print("Generated demographic distribution overall:")
display(
    customer_import_audit[["income_band", "education_level", "age_band", "marital_status", "gender", "home_owner"]]
    .melt(var_name="field", value_name="value")
    .groupby(["field", "value"])
    .size()
    .rename("count")
    .reset_index()
    .assign(pct=lambda frame: frame["count"] / len(customer_import_audit))
)

example_zips = ["77494", "77002", "75201", "78701", "79936"]
for example_zip in example_zips:
    example_rows = customer_import_audit.loc[customer_import_audit["zip_code"].eq(example_zip)]
    if example_rows.empty:
        print(f"ZIP {example_zip}: no generated customers")
        continue
    print(f"ZIP {example_zip}: {len(example_rows):,} customers")
    display(
        example_rows[["income_band", "education_level", "age_band", "home_owner", "marital_status"]]
        .melt(var_name="field", value_name="value")
        .groupby(["field", "value"])
        .size()
        .rename("count")
        .reset_index()
    )

print("Customers by city/county:")
display(
    customer_import_audit.groupby(["city", "assigned_texas_county"], dropna=False)
    .size()
    .rename("customer_count")
    .reset_index()
    .sort_values("customer_count", ascending=False)
    .head(30)
)

print("Top 20 ZIPs by customer count:")
display(
    customer_import_audit.groupby("zip_code")
    .size()
    .rename("customer_count")
    .reset_index()
    .sort_values("customer_count", ascending=False)
    .head(20)
)

## Final Notes

This notebook uses ACS ZCTA targets as local probability controls. Each customer keeps the assigned Texas ZIP, city, and county from `postal_customers.csv`. The demographic fields are generated so each ZIP roughly follows its ACS income, education, age, homeownership, and married-household profile.

The CSV stores `customer_id` as 32-character hex text because MySQL stores `customer_id` as `BINARY(16)`. The SQL load script converts it with `UNHEX(customer_id)`.